<a href="https://colab.research.google.com/github/DonMilcrypto/sketch2img/blob/feature-high-quality-improvements/Sketch_to_Image_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sketch-to-Image Generator on Google Colab

This notebook allows you to run the complete Sketch-to-Image Generator application in a Google Colab environment.

**Instructions:**
1.  **Add your API Key:** Click the **key icon** on the left sidebar to open the "Secrets" tab. Create a new secret with the name `HF_TOKEN` and paste your Hugging Face API key as the value.
2.  **Run the cells:** Execute the cells in order by clicking the play button on each one. The final cell will start the application and provide a public URL.

In [20]:
#@title This cell will install all the required Python packages.
!pip install gradio requests Pillow numpy opencv-python datasets python-dotenv --quiet
print("✅ Dependencies installed successfully.")

✅ Dependencies installed successfully.


In [21]:
#@title Write image_generator.py
%%writefile image_generator.py
# image_generator.py
"""
Backend logic for the Sketch-to-Image Generator application.
"""
import os, time, asyncio, requests, base64, logging
from io import BytesIO
from PIL import Image
import numpy as np
import cv2
import config

# The API key will be set as an environment variable directly from Colab secrets
HUGGINGFACE_API_KEY = os.getenv("HF_TOKEN")
if not HUGGINGFACE_API_KEY:
    logging.warning("Hugging Face API Key (HF_TOKEN) not set. Image generation will fail.")

def numpy_to_pil(numpy_array):
    if numpy_array is None: return None
    try:
        if len(numpy_array.shape) == 3 and numpy_array.shape[2] in [3, 4]:
            return Image.fromarray(numpy_array[:, :, :3].astype(np.uint8), 'RGB')
        elif len(numpy_array.shape) == 2:
            return Image.fromarray(numpy_array.astype(np.uint8), 'L').convert('RGB')
    except Exception as e:
        logging.error(f"Error converting numpy to PIL: {e}", exc_info=True)
    return None

def pil_to_numpy(pil_image):
    if pil_image is None: return None
    try:
        if pil_image.mode != 'RGB': pil_image = pil_image.convert('RGB')
        return np.array(pil_image)
    except Exception as e:
        logging.error(f"Error converting PIL to numpy: {e}", exc_info=True)
    return None

def analyze_drawing(pil_image):
    shapes, dominant_colors = [], []
    if pil_image is None: return shapes, dominant_colors
    try:
        gray = cv2.cvtColor(pil_to_numpy(pil_image), cv2.COLOR_RGB2GRAY)
        _, thresh = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) > config.MIN_CONTOUR_AREA:
                perimeter = cv2.arcLength(contour, True)
                approx = cv2.approxPolyDP(contour, config.SHAPE_APPROX_EPSILON * perimeter, True)
                shapes.append(_classify_shape(approx))
        dominant_colors = _extract_colors(thresh, pil_to_numpy(pil_image))
    except Exception as e:
        logging.error(f"Error analyzing drawing: {e}", exc_info=True)
    return shapes, dominant_colors

def _classify_shape(approx):
    num_vertices = len(approx)
    if num_vertices == 3: return "triangle"
    elif num_vertices == 4: return "rectangle"
    elif num_vertices > 4:
        area, perimeter = cv2.contourArea(approx), cv2.arcLength(approx, True)
        if perimeter == 0: return "polygon"
        circularity = 4 * np.pi * (area / (perimeter * perimeter))
        return "circle" if config.CIRCLE_CIRCULARITY_THRESHOLD[0] < circularity < config.CIRCLE_CIRCULARITY_THRESHOLD[1] else "polygon"
    return "unknown"

def _extract_colors(thresh, numpy_image_rgb):
    if numpy_image_rgb is None: return []
    try:
        mask = cv2.cvtColor(thresh, cv2.COLOR_GRAY2RGB) > 0
        pixels = numpy_image_rgb[mask]
        unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)
        sorted_indices = np.argsort(counts)[::-1]
        return [f"#{color[0]:02x}{color[1]:02x}{color[2]:02x}" for color in unique_colors[sorted_indices[:5]]]
    except Exception as e:
        logging.error(f"Error extracting colors: {e}", exc_info=True)
    return []

async def generate_image(pil_image, prompt_override="", selected_model="Stable Diffusion XL Turbo", negative_prompt="", guidance_scale=7.5, seed=-1):
    if pil_image is None: return None, "Draw something on the canvas."
    if not HUGGINGFACE_API_KEY: return None, "API key not set."
    model_id = config.AVAILABLE_MODELS.get(selected_model, list(config.AVAILABLE_MODELS.values())[0])
    shapes, colors = analyze_drawing(pil_image)
    prompt = prompt_override.strip() or f"a sketch featuring {', '.join(shapes)} with colors {', '.join(colors)}"
    headers = {"Authorization": f"Bearer {HUGGINGFACE_API_KEY}"}
    parameters = {"negative_prompt": negative_prompt.strip() or "low quality, blurry", "guidance_scale": guidance_scale}
    if seed != -1: parameters["seed"] = seed
    payload = {"inputs": prompt, "parameters": parameters, "options": {"wait_for_model": True}}
    try:
        api_url = f"https://api-inference.huggingface.co/models/{model_id}"
        response = await asyncio.to_thread(requests.post, api_url, headers=headers, json=payload, timeout=config.REQUEST_TIMEOUT)
        response.raise_for_status()
        if "image/" in response.headers.get("Content-Type", ""): return Image.open(BytesIO(response.content)), "Image generated successfully."
    except requests.exceptions.RequestException as e:
        error_message = e.response.json().get("error", str(e)) if e.response else str(e)
        return None, f"API Error: {error_message}"
    return None, "Failed to generate image."

async def inpaint_image(image_dict, prompt):
    if not image_dict or "image" not in image_dict or "mask" not in image_dict: return None, "Missing image or mask for inpainting."
    if not HUGGINGFACE_API_KEY: return None, "API key not set."
    image, mask = image_dict["image"], image_dict["mask"]
    buffered_img = BytesIO(); image.save(buffered_img, format="PNG"); img_str = base64.b64encode(buffered_img.getvalue()).decode()
    buffered_mask = BytesIO(); mask.save(buffered_mask, format="PNG"); mask_str = base64.b64encode(buffered_mask.getvalue()).decode()
    headers = {"Authorization": f"Bearer {HUGGINGFACE_API_KEY}"}
    payload = {"inputs": prompt, "parameters": {"image": img_str, "mask_image": mask_str}, "options": {"wait_for_model": True}}
    try:
        api_url = f"https://api-inference.huggingface.co/models/{config.INPAINTING_MODEL}"
        response = await asyncio.to_thread(requests.post, api_url, headers=headers, json=payload, timeout=config.REQUEST_TIMEOUT)
        response.raise_for_status()
        if "image/" in response.headers.get("Content-Type", ""): return Image.open(BytesIO(response.content)), "Inpainting successful."
    except requests.exceptions.RequestException as e:
        error_message = e.response.json().get("error", str(e)) if e.response else str(e)
        return None, f"API Error: {error_message}"
    return None, "Failed to inpaint image."

Overwriting image_generator.py


In [22]:
#@title This cell writes the configuration code to the Colab environment.
%%writefile config.py
# config.py
"""
Configuration settings for the Sketch-to-Image Generator application.
"""
AVAILABLE_MODELS = {
    "Stable Diffusion XL Turbo": "stabilityai/stable-diffusion-xl-turbo",
    "Stable Diffusion 2.1": "stabilityai/stable-diffusion-2-1",
    "Stable Diffusion 2 Inpainting": "stabilityai/stable-diffusion-2-inpainting",
}
INPAINTING_MODEL = "stabilityai/stable-diffusion-2-inpainting"
REQUEST_TIMEOUT = 90
THROTTLE_TIME = 1.5
MIN_CONTOUR_AREA = 100
SHAPE_APPROX_EPSILON = 0.02
CIRCLE_CIRCULARITY_THRESHOLD = (0.6, 1.4)

Overwriting config.py


In [26]:
%%writefile app.py
# app.py
"""
UI for the Sketch-to-Image Generator.
"""
import gradio as gr
import logging
import config
from image_generator import generate_image, numpy_to_pil, inpaint_image

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def create_gradio_interface():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Sketch-to-Image Generator")
        with gr.Tabs():
            with gr.TabItem("Create"):
                with gr.Row():
                    canvas = gr.Image(label="Draw or Upload Image", type="numpy", image_mode="RGB", height=512, width=512, interactive=True)
                    output_image = gr.Image(label="Generated Image", interactive=False, height=512, width=512)
                prompt_input = gr.Textbox(label="Text Prompt", placeholder="Describe the image...")
                with gr.Accordion("Generation History", open=True):
                    history_gallery = gr.Gallery(label="Recent creations", height=256)
            with gr.TabItem("Advanced Settings"):
                negative_prompt_input = gr.Textbox(label="Negative Prompt", placeholder="Describe what you DON'T want...")
                guidance_scale_slider = gr.Slider(minimum=0, maximum=20, step=0.5, value=7.5, label="Guidance Scale (CFG)")
                seed_input = gr.Number(label="Seed", value=-1, precision=0, interactive=True)
            with gr.TabItem("Inpainting"):
                with gr.Row():
                    inpainting_image = gr.Image(label="Image to Edit", type="pil", height=512, width=512)
                    inpainting_output = gr.Image(label="Inpainted Result", interactive=False, height=512, width=512)
                inpainting_prompt = gr.Textbox(label="Inpainting Prompt", placeholder="Describe the new content...")
                inpainting_button = gr.Button("Regenerate Masked Area", variant="primary")
        status_display = gr.Textbox(label="Status", interactive=False, lines=1)
        generate_button = gr.Button("Generate Image", variant="primary")
        history_state = gr.State([])
        async def on_generate(image_dict, prompt, negative_prompt, guidance_scale, seed, current_history):
            if image_dict is None: return None, "Please draw or upload an image.", current_history
            pil_image = numpy_to_pil(image_dict)
            generated_img, status = await generate_image(pil_image, prompt, "Stable Diffusion XL Turbo", negative_prompt, guidance_scale, int(seed))
            if generated_img:
                current_history.insert(0, generated_img)
                if len(current_history) > 10: current_history.pop()
            return generated_img, status, current_history
        generate_button.click(fn=on_generate, inputs=[canvas, prompt_input, negative_prompt_input, guidance_scale_slider, seed_input, history_state], outputs=[output_image, status_display, history_gallery])
        inpainting_button.click(fn=inpaint_image, inputs=[inpainting_image, inpainting_prompt], outputs=[inpainting_output, status_display])
    return demo

if __name__ == "__main__":
    app_interface = create_gradio_interface()
    app_interface.launch(share=True)

print("✅ Application files created successfully!")

Overwriting app.py


In [24]:
#@title This cell will load your Hugging Face API key from Colab's secrets and then start the Gradio web application.

import os
from google.colab import userdata

# Load the API key from Colab secrets and set it as an environment variable
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("✅ Successfully loaded Hugging Face API key.")
except userdata.SecretNotFoundError:
    print("⚠️ ERROR: 'HF_TOKEN' not found in Colab secrets. Please go to the 'Secrets' tab (key icon on the left) and add it.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Run the Gradio application
# The output will include a public URL that you can click to open the app.
!python3 app.py

✅ Successfully loaded Hugging Face API key.
Traceback (most recent call last):
  File "/content/app.py", line 49, in <module>
    app_interface = create_gradio_interface()
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/app.py", line 18, in create_gradio_interface
    canvas = gr.Image(label="Draw or Upload Image", type="numpy", image_mode="RGB", height=512, width=512, interactive=True, sources=["canvas", "upload"])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/component_meta.py", line 189, in wrapper
    return fn(self, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/components/image.py", line 177, in __init__
    raise ValueError(
ValueError: `sources` must a list consisting of elements in ['upload', 'webcam', 'clipboard']
2025-11-18 06:05:24,963 - INFO - HTTP R

In [ ]:
#@title This cell will load your Hugging Face API key from Colab's secrets and then start the Gradio web application.

import os
from google.colab import userdata

# Load the API key from Colab secrets and set it as an environment variable
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("✅ Successfully loaded Hugging Face API key.")
except userdata.SecretNotFoundError:
    print("⚠️ ERROR: 'HF_TOKEN' not found in Colab secrets. Please go to the 'Secrets' tab (key icon on the left) and add it.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Run the Gradio application
# The output will include a public URL that you can click to open the app.
!python3 app.py

✅ Successfully loaded Hugging Face API key.
* Running on local URL:  http://127.0.0.1:7860
2025-11-18 06:08:35,545 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-11-18 06:08:35,562 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2025-11-18 06:08:35,821 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2025-11-18 06:08:36,140 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
* Running on public URL: https://33acf37c86c36ba3b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^